In [2]:
import birdnet
from datetime import datetime
import sys
import dotenv
import os
import pandas as pd
import matplotlib.pyplot as plt

from database import DetectionService, Engine

## Step 1: get recordings from the database

In [ ]:
# Open and load the config
try:
    dotenv.load_dotenv('/etc/bird_audio_pipeline.conf')
except FileNotFoundError:
    print("ERROR: Can't find config file")
    sys.exit(0)

def get_db_credentials_dict():
    db_credentials_dict = {'user': os.getenv('DATABASE_USER'),
                           'password': os.getenv('DATABASE_PASSWORD'),
                           'database': os.getenv('DATABASE_NAME')}
    return db_credentials_dict

db_credentials = get_db_credentials_dict()
db_engine = Engine(db_credentials)

deter_serv = DetectionService(db_engine.engine)

df = deter_serv.get_recordings()



## Step 2: Load the birdnet model

In [3]:
model_version = "2.4"
# Load the official BirdNET acoustic model
model = birdnet.load("acoustic", model_version, "tf", lang='nl')

# Load the geo model
geo_model = birdnet.load("geo", model_version, "tf", lang='nl')

## Step 3: Tweaking the minimal confidence parameter for geo predictions
The predictions for the species list have a cut-off parameter for the confidence scores. Any prediction below that value will not be returned. A logical thing would be to set this fairly high to get better detections down the line, but I want to see how low I get. I want to do this for several reasons:
1. Rare sightings can occur, and I want to detect those.
2. The predictions are based on observations submitted to eBird. While the coverage is high in Europe, I have a slight mistrust in data with the user as a source.

I think the value 0.01 will serve as a suitable minimal confidence score, but I want to make sure. I will make four geo models and plot the results
- Four different weeks: 1, 14, 27 and 40
- No minimal confidence. This will include the entire taxonomy of BirdNET
- The same longitude and latitude for all predictions
- Species with an extremely low confidence will not be shown in the plot for better scaling

In [ ]:
def make_plot(geo_df, week_number):
    threshold = 0.0001
    plot_df = (
        geo_df.loc[geo_df["confidence"] >= threshold]
        .sort_values("confidence", ascending=False)
        .reset_index(drop=True)
    )

    plt.figure(figsize=(20, 10))
    plt.plot(plot_df.index, plot_df["confidence"], marker=".", linewidth=1)

    plt.axhline(
        y=0.01,
        color="red",
        linestyle="--",
        linewidth=1.5,
        label="0.01 confidence",
    )

    plt.xlabel("Species (sorted by confidence)")
    plt.ylabel("Confidence score")
    plt.title(f"Confidence-score flow (week= {week_number})")
    plt.xlim(0, len(plot_df) - 1)
    plt.grid(axis="y", alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

weeks_to_test = [1, 14, 27, 40]
for week in weeks_to_test:
    geo_predictions_test = geo_model.predict(
        52.905531,
        6.626824,
        week=week,
        min_confidence=0.00
    )

    geo_df = geo_predictions_test.to_dataframe().sort_values('confidence', ascending=False)
    make_plot(geo_df, week)


All plots show the same rapid decline in confidence scores. When setting the minimal score at 0.01, I will still get a lot of species, but the bulk of the predictions will be omitted. These are near zero and will only increase the chance of false detections and the runtime per file. I do recognize the resulting model still contains a large amount of species. I will use their low predictions to flag detections of species with low confidences.

## Step 4: Make predictions with different overlap parameters

The BirdNET model analyzes audio by dividing it in windows of 3 seconds. While this base value is set, the model offers an overlap parameter. This will determine where each window will start.
- An overlap of zero will result in windows: 0-3, 4-6, 7-9, etc.
- An overlap of 2 will result in windows: 0-3, 1-4, 2-5, etc.

The following code will loop over a range of possible overlap values to determine the optimal value.

In [ ]:
#dataframe to append the results to
results_df = pd.DataFrame(columns=['file_path',
                                   'overlap_value',
                                   'observation_count',
                                   'species_count',
                                   'median_confidence',
                                   'max_confidence',
                                   'min_confidence',
                                   'processing_time',
                                   'model_ver'])
date_file_name = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
use_header = True

The same recordings will now be subjected to the model and its results will be saved to a file.

In [ ]:
for row in df.iterrows():
    data = row[1]
    date = data['rec_date']
    week_number = int(date.isocalendar()[1])

    lat = float(data['latitude'])
    lon = float(data['longitude'])

    geo_predictions = geo_model.predict(
    lat,
    lon,
    week=week_number,
    min_confidence=0.01
    )

    print(geo_predictions.to_dataframe())

    print(f'Prediction for {data['file_path']}')

    overlap_val = [0, 0.5, 1, 1.5, 2.0, 2.5]

    for val in overlap_val:
        print(f'Overlap value: {val}')
        start_time = datetime.now()
        # Predict species
        predictions = model.predict(
            data['file_path'],
            custom_species_list=geo_predictions.to_set(),
            n_workers=8,
            batch_size=16,
            overlap_duration_s=val
        )
        predictions_df = predictions.to_dataframe()

        observation_count = len(predictions_df)
        species_count = predictions_df['species_name'].nunique()

        stop_time = datetime.now()
        runtime = (stop_time - start_time).total_seconds()

        highest_confidence = predictions_df['confidence'].max()
        lowest_confidence = predictions_df['confidence'].min()
        median_confidence = predictions_df['confidence'].median()

        results_dict = {'file_path': data['file_path'],
                        'overlap_value': val,
                        'observation_count': observation_count,
                        'species_count': species_count,
                        'median_confidence': median_confidence,
                        'max_confidence': highest_confidence,
                        'min_confidence': lowest_confidence,
                        'processing_time': runtime,
                        'model_ver': model_version,}

        results_df = pd.concat([results_df, pd.DataFrame([results_dict])], ignore_index=True)

    print('Processed file: {}'.format(data['file_path']))
    print(results_df.head(3))
    results_df.to_csv(f'/home/tom/ResearchData/Microfoon_onderzoek/parameter_checks/parameter_check_{date_file_name}.tsv', index=False, sep='\t', mode='a', header=use_header)
    use_header = False
    
print('Done')


Plots will be made to visualize the results.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd


def plot_overlap_summary(df):
    # Average and standard deviation per overlap
    summary = (
        df.groupby("overlap_value")
          .agg(
              observations_mean=("observation_count", "mean"),
              observations_std=("observation_count", "std"),
              species_mean=("species_count", "mean"),
              species_std=("species_count", "std"),
              runtime_mean=("processing_time", "mean"),
              runtime_std=("processing_time", "std"),
              median_confidence=("median_confidence", "mean"),
              min_confidence=("min_confidence", "mean"),
              max_confidence=("max_confidence", "mean"),
          )
          .reset_index()
    )

    fig, axs = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)

    # ----------------------------------------------------
    # Observations & species
    # ----------------------------------------------------
    ax = axs[0, 0]

    ax.errorbar(
        summary["overlap_value"],
        summary["observations_mean"],
        yerr=summary["observations_std"],
        marker="o",
        capsize=3,
        label="Observations",
    )

    ax.errorbar(
        summary["overlap_value"],
        summary["species_mean"],
        yerr=summary["species_std"],
        marker="s",
        capsize=3,
        label="Species",
    )

    ax.set_title("Detections")
    ax.set_xlabel("Overlap (s)")
    ax.set_ylabel("Count")
    ax.grid(alpha=0.3)
    ax.legend()

    # ----------------------------------------------------
    # Runtime
    # ----------------------------------------------------
    ax = axs[0, 1]

    ax.errorbar(
        summary["overlap_value"],
        summary["runtime_mean"],
        yerr=summary["runtime_std"],
        marker="o",
        capsize=3,
    )

    ax.set_title("Processing time")
    ax.set_xlabel("Overlap (s)")
    ax.set_ylabel("Seconds")
    ax.grid(alpha=0.3)

    # ----------------------------------------------------
    # Confidence
    # ----------------------------------------------------
    ax = axs[1, 0]

    ax.plot(summary["overlap_value"], summary["median_confidence"],
            marker="o", label="Median")

    ax.plot(summary["overlap_value"], summary["min_confidence"],
            marker="s", label="Minimum")

    ax.plot(summary["overlap_value"], summary["max_confidence"],
            marker="^", label="Maximum")

    ax.set_ylim(0, 1)
    ax.set_title("Confidence")
    ax.set_xlabel("Overlap (s)")
    ax.set_ylabel("Confidence")
    ax.grid(alpha=0.3)
    ax.legend()

    # ----------------------------------------------------
    # Observations per species
    # ----------------------------------------------------
    ax = axs[1, 1]

    ratio = summary["observations_mean"] / summary["species_mean"]

    ax.plot(summary["overlap_value"], ratio,
            marker="o")

    ax.set_title("Observations per species")
    ax.set_xlabel("Overlap (s)")
    ax.set_ylabel("Ratio")
    ax.grid(alpha=0.3)


    plt.show()

overlap_test_results = pd.read_csv('/home/tom/ResearchData/Microfoon_onderzoek/parameter_checks/parameter_check_2026-07-13_21-55-55.tsv', delimiter='\t', header=0)
plot_overlap_summary(overlap_test_results)

It's clear that after a certain overlap, the number of detections is rapidly increasing. However, the number of species remains the same. This indicates that the model does not hear more types of bird at a certain overlap, and a saturation will be reached. It simply hears the same bird again and again. After a while, all it does is increase processing time.
An overlap of 1.5 s was selected because it represented a favorable compromise between detection performance and computational cost. Larger overlaps produced substantially more detections and processing time while yielding comparatively smaller gains in detected species.

## Conclusion

**Geo model minimal confidence threshold:** 0.01
**Overlap parameter:** 1.5